# SQL Refresher 4

## imports

In [1]:
import math
import numpy as np
import pandas as pd

import psycopg2

import os
import json

import gmaps
import gmaps.geojson_geometries

from geographiclib.geodesic import Geodesic

## my_select_query_pandas() - function to run a select query and return rows in a Pandas dataframe

In [2]:
#
# function to run a select query and return rows in a pandas dataframe
# pandas puts all numeric values from postgres to float
# if it will fit in an integer, change it to integer
#

def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)
    

## Connect to the Postgres database

In [3]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

## Create a cursor for the connection

In [4]:
cursor = connection.cursor()

# Lab: Google Maps - Display Map by Type, Size, Zoom

## Connect to Google Maps using your api key;  edit the file gmap_api_key.txt and put in your api key

In [6]:
f = open('gmap_api_key.txt', 'r')
# f = open('labs/week_02/gmap_api_key.txt', 'r')

my_api_key = f.read()
f.close()

gmaps.configure(api_key=my_api_key)

## Basic map centered on Sather Gate at UC Berkeley; all point are (latitude, longitude) in decimal; zoom levels go from 1 to 21; 1 is world level; 21 is street level; default type of map is Road Map

In [7]:
sather_gate_berkeley = (37.870260430419115, -122.25950168579497)

gmaps.figure(center=sather_gate_berkeley, zoom_level=9)

Figure(layout=FigureLayout(height='420px'))

## Same map as Terrain

In [8]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=9, map_type='TERRAIN')

Figure(layout=FigureLayout(height='420px'))

## Same map as Satellite

In [9]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=9, map_type='SATELLITE')

Figure(layout=FigureLayout(height='420px'))

## Same map as Hybrid (combines Road Map with Satellite)

In [10]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=9, map_type='HYBRID')

Figure(layout=FigureLayout(height='420px'))

## Same map with the sizes changed

In [11]:
figure_layout = {
    'width': '300px',
    'height': '500px',
    'border': '1px solid black',
    'padding': '1px'
}

gmaps.figure(center=sather_gate_berkeley, zoom_level=9, layout=figure_layout)

Figure(layout=FigureLayout(border='1px solid black', height='500px', padding='1px', width='300px'))

## Same map with zoom level 12 (zoom goes from 1 to 21, 1 is world, 21 is street)

In [12]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=12)

Figure(layout=FigureLayout(height='420px'))

## Same map with zoom level 15 (zoom goes from 1 to 21, 1 is world, 21 is street)

In [13]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=15)

Figure(layout=FigureLayout(height='420px'))

## Same map with zoom level 17 (zoom goes from 1 to 21, 1 is world, 21 is street)

In [14]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=17)

Figure(layout=FigureLayout(height='420px'))

## Same map with zoom level 19 (zoom goes from 1 to 21, 1 is world, 21 is street)

In [15]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=19)

Figure(layout=FigureLayout(height='420px'))

## Same map with zoom level 21 (zoom goes from 1 to 21, 1 is world, 21 is street)

In [16]:
gmaps.figure(center=sather_gate_berkeley, zoom_level=21)

Figure(layout=FigureLayout(height='420px'))

# Lab: Google Maps - Location Markers, Location Symbols

## Find a list of zip codes for all customers of the Berkeley store; distinct removes duplicates

In [17]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select distinct z.*
from customers as cu
     join zip_codes as z
         on cu.zip = z.zip
where cu.closest_store_id = 1
order by 1,2

"""

df = my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)
df

,zip,latitude,longitude,city,state,population,area,density,time_zone
0,94002,37.5135,-122.2991,Belmont,CA,27202,5.9244,4591.53,America/Los_Angeles
1,94005,37.6887,-122.4080,Brisbane,CA,4692,4.8168,974.09,America/Los_Angeles
2,94010,37.5693,-122.3653,Burlingame,CA,42730,12.4205,3440.28,America/Los_Angeles
3,94014,37.6909,-122.4475,Daly City,CA,49515,6.6361,7461.50,America/Los_Angeles
4,94015,37.6812,-122.4805,Daly City,CA,64887,6.1247,10594.35,America/Los_Angeles
...,...,...,...,...,...,...,...,...,...
139,94963,38.0138,-122.6703,San Geronimo,CA,498,2.3307,213.67,America/Los_Angeles
140,94964,37.9431,-122.4918,San Quentin,CA,3418,0.2608,13104.83,America/Los_Angeles
141,94965,37.8499,-122.5236,Sausalito,CA,11408,14.2131,802.64,America/Los_Angeles
142,94970,37.9145,-122.6469,Stinson Beach,CA,689,7.0006,98.42,America/Los_Angeles


## Same map with a marker added for the center of each zip code that has customers for the Berkeley store;  clicking on a marker will not do anything

In [18]:
fig = gmaps.figure(center=sather_gate_berkeley, zoom_level=8)

df_markers = df[['latitude','longitude']]

marker_layer = gmaps.marker_layer(df_markers)

fig.add_layer(marker_layer)

fig

Figure(layout=FigureLayout(height='420px'))

## Same map adding information about each marker;  when you click on a marker, a popup will show you information about that zip code

In [19]:
# Adding info box to markers

info_box_template = """
<dl>
<dt>Zip</dt><dd>{zip}</dd>
<dt>Latitude</dt><dd>{latitude}</dd>
<dt>Longitude</dt><dd>{longitude}</dd>
<dt>City</dt><dd>{city}</dd>
<dt>State</dt><dd>{state}</dd>
<dt>Population</dt><dd>{population}</dd>
<dt>Area</dt><dd>{area}</dd>
<dt>Density</dt><dd>{density}</dd>
<dt>Time Zone</dt><dd>{time_zone}</dd>

<dt>Population</dt><dd>{population}</dd>
</dl>
"""

df_list_dict = df.to_dict('records')

zip_info = [info_box_template.format(**zip) for zip in df_list_dict]

marker_layer = gmaps.marker_layer(df_markers, info_box_content=zip_info)

fig = gmaps.figure(center=sather_gate_berkeley, zoom_level=8)

fig.add_layer(marker_layer)

fig


Figure(layout=FigureLayout(height='420px'))

# Lab: Google Maps - Heatmaps, Choropleths

## Find a list of zip codes for customers of the Berkeley store; do not remove duplicates so we can use intensity levels for each zip code

In [20]:
rollback_before_flag = True
rollback_after_flag = True

# We didn't use distinct to remove dups so we could get the "density"
query = """

select z.latitude, z.longitude
from customers as cu
     join zip_codes as z
         on cu.zip = z.zip
where cu.closest_store_id = 1
order by 1,2

"""

df = my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)
df

,latitude,longitude
0,37.4949,-122.2080
1,37.4949,-122.2080
2,37.4949,-122.2080
3,37.4949,-122.2080
4,37.4949,-122.2080
...,...,...
8133,38.1842,-122.2629
8134,38.1842,-122.2629
8135,38.1842,-122.2629
8136,38.1842,-122.2629


## Heatmap showing intensity as the number of customers in each zip code

In [21]:
fig = gmaps.figure(center=sather_gate_berkeley, zoom_level=8)

heatmap_layer = gmaps.heatmap_layer(df)

fig.add_layer(heatmap_layer)

fig

Figure(layout=FigureLayout(height='420px'))

## Same map but in Hybrid;  a lot of people think heat maps look better with Hybrid instead of Road Map

In [22]:
fig = gmaps.figure(center=sather_gate_berkeley, map_type = 'HYBRID', zoom_level=8)

heatmap_layer = gmaps.heatmap_layer(df)

fig.add_layer(heatmap_layer)

fig

Figure(layout=FigureLayout(height='420px'))

## Find a list of geojson geometries that are built into Google Maps;  note that us zip codes are not in there

In [23]:
gmaps.geojson_geometries.list_geometries()


dict_keys(['countries', 'countries-high-resolution', 'england-counties', 'us-states', 'us-counties', 'india-states', 'brazil-states'])

## Load the geometry for US States;  give each state a random color

In [24]:
states_geojson = gmaps.geojson_geometries.load_geometry('us-states')

fig = gmaps.figure(center=sather_gate_berkeley, zoom_level=2)

colors = [np.random.choice(['red', 'green', 'blue', 'purple', 'yellow', 'teal']) for state in states_geojson['features']]

geojson_layer = gmaps.geojson_layer(states_geojson, fill_color=colors)

fig.add_layer(geojson_layer)

fig

Figure(layout=FigureLayout(height='420px'))

## If you want a US zip code geometry, it's not built into Google Maps, you have to download it from other sources; the directory geojson_data has these files downloaded for you;  customer_zip_geojson.json is just the zip codes for our customers for all stores

In [25]:
# Kevin kinedly curated this dataset for us
f = open('labs/week_02/geojson_data/customer_zip_geojson.json')
customer_zip_geojson = json.load(f)
f.close()

kansas = (38.4942,-98.3132)

fig = gmaps.figure(center=kansas, zoom_level=4)

colors = [np.random.choice(['red', 'green', 'blue', 'purple', 'yellow', 'teal']) 
          for state in customer_zip_geojson['features']]
          
geojson_layer = gmaps.geojson_layer(customer_zip_geojson, fill_color=colors)

fig.add_layer(geojson_layer)

fig

FileNotFoundError: [Errno 2] No such file or directory: 'labs/week_02/geojson_data/customer_zip_geojson.json'

# Lab: Google Maps - Driving Directions, Traffic Layers, Transit Layers

## Bicycling Layer

In [26]:
# Bicycle trails
fig = gmaps.figure(center=sather_gate_berkeley, zoom_level=9)

fig.add_layer(gmaps.bicycling_layer())

fig

Figure(layout=FigureLayout(height='420px'))

## Transit Layer

In [27]:
# The subways?
fig = gmaps.figure(center=sather_gate_berkeley, zoom_level=10)

fig.add_layer(gmaps.transit_layer())

fig

Figure(layout=FigureLayout(height='420px'))

## Traffic Layer

In [28]:
# Looks busy
fig = gmaps.figure(center=sather_gate_berkeley, zoom_level=10)

fig.add_layer(gmaps.traffic_layer())

fig

Figure(layout=FigureLayout(height='420px'))

## Directions from Sather Gate at UC Berkeley to the Computer History Museum

In [29]:
midpoint = (37.6750258153003, -122.22092635822978)

computer_history_museum = (37.41431101165934, -122.0771717192328)

fig = gmaps.figure(center=midpoint, zoom_level=9)

sather_2_chm = gmaps.directions_layer(sather_gate_berkeley, computer_history_museum)

fig.add_layer(sather_2_chm)

fig

Figure(layout=FigureLayout(height='420px'))

## my_calculate_distance() - function to calculate the geodesic distance in miles between two points;  unlike great circle, geodesic takes into account that the earth is an ellipsoid not a sphere

In [30]:
#
#  Given two points in (latitude, longitude) format, calculate the distance between them in miles
#

def my_calculate_distance(point_1, point_2):
    "Given two points in (latitude, longitude) format, calculate the distance between them in miles"
    
    geod = Geodesic.WGS84


    g = geod.Inverse(point_1[0], point_1[1], point_2[0], point_2[1])
    miles = g['s12'] / 1000 * 0.621371
    
    return miles

In [31]:
my_calculate_distance(sather_gate_berkeley, computer_history_museum)

32.996404285783115

## Map with the geodesic distance between Sather Gate at UC Berkeley and the Computer History Museum;  compare and contrast to the Directions map above

In [32]:
fig = gmaps.figure(center=midpoint, zoom_level=9)

lines = []

lines.append(gmaps.Line(start=sather_gate_berkeley, end=computer_history_museum, stroke_color='blue'))

drawing_layer = gmaps.drawing_layer(features=lines)

fig.add_layer(drawing_layer)

fig

Figure(layout=FigureLayout(height='420px'))

## my_calculate_new_point() - function that takes a point, direction, and miles and find a new point;  direction is in degrees: 0 = North, 90 = East, 180 = South, 270 = West

In [33]:
#
#  Given a point, direction, and miles, calculate a new point
#
#  Direction is in degrees, with 0 = North, 90 = East, 180 = South, 270 = West
#
#

def my_calculate_new_point(point, direction, miles):
    "Given a point, direction, and miles, calculate the box in form left, right, top, bottom"
    
    geod = Geodesic.WGS84

    kilometers = miles * 1.60934
    meters = kilometers * 1000

    g = geod.Direct(point[0], point[1], direction, meters)
    
    new_point = (g['lat2'], g['lon2'])
    
    return(new_point)

## Draw a line from Sather Gate at UC Berkeley to a point 100 miles North East

In [34]:
# find points with distance
north_east = 45
miles = 100

midpoint = (38.518434,-121.576177)

new_point = my_calculate_new_point(sather_gate_berkeley, north_east, miles)

fig = gmaps.figure(center=(38.518434,-121.576177), zoom_level=7)

lines = []

lines.append(gmaps.Line(start=sather_gate_berkeley, end=new_point, stroke_color='blue'))

drawing_layer = gmaps.drawing_layer(features=lines)

fig.add_layer(drawing_layer)

fig

Figure(layout=FigureLayout(height='420px'))

## my_calculate_box() - function that take a point and X miles and finds a box;  left is west X miles; right is X miles east; top is X miles north; bottom is X miles south

In [35]:
#
#  Given a point and miles, calculate the box  
#
#

def my_calculate_box(point, miles):
    "Given a point and miles, calculate the box in form left, right, top, bottom"
    
    geod = Geodesic.WGS84

    kilometers = miles * 1.60934
    meters = kilometers * 1000

    g = geod.Direct(point[0], point[1], 270, meters)
    left = (g['lat2'], g['lon2'])

    g = geod.Direct(point[0], point[1], 90, meters)
    right = (g['lat2'], g['lon2'])

    g = geod.Direct(point[0], point[1], 0, meters)
    top = (g['lat2'], g['lon2'])

    g = geod.Direct(point[0], point[1], 180, meters)
    bottom = (g['lat2'], g['lon2'])
    
    return(left, right, top, bottom)

In [36]:
left, right, top, bottom = my_calculate_box(sather_gate_berkeley, 5)

print (left, right, top, bottom)

(37.8702249126905, -122.35095496602143) (37.8702249126905, -122.16804840556851) (37.9427566793502, -122.25950168579497) (37.79776328649186, -122.25950168579497)


## The box allows us to easily find point stored in a database table

In [37]:

connection.rollback()

query = "select zip from zip_codes "
query += " where latitude >= " + str(bottom[0])
query += " and latitude <= " + str(top [0])
query += " and longitude >= " + str(left[1])
query += " and longitude <= " + str(right[1])
query += " order by 1 "

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    print(row[0])


94530
94563
94602
94607
94608
94609
94610
94611
94612
94618
94702
94703
94704
94705
94706
94707
94708
94709
94710
94720
94804


## To draw the box on the map, we need to change the format of the box to find the corners

In [38]:
# find corners

corners = [ (top[0], left[1]), (top[0], right[1]), (bottom[0], right[1]), (bottom[0], left[1])]

corners

[(37.9427566793502, -122.35095496602143),
 (37.9427566793502, -122.16804840556851),
 (37.79776328649186, -122.16804840556851),
 (37.79776328649186, -122.35095496602143)]

## Draw the outline of the box by drawing 4 lines between the 4 corners

In [39]:
fig = gmaps.figure(center=sather_gate_berkeley, zoom_level=10)

lines = []

lines.append(gmaps.Line(start=corners[0], end=corners[1], stroke_color='blue'))
lines.append(gmaps.Line(start=corners[1], end=corners[2], stroke_color='blue'))
lines.append(gmaps.Line(start=corners[2], end=corners[3], stroke_color='blue'))
lines.append(gmaps.Line(start=corners[3], end=corners[0], stroke_color='blue'))

drawing_layer = gmaps.drawing_layer(features=lines)

fig.add_layer(drawing_layer)

fig

Figure(layout=FigureLayout(height='420px'))

## Draw the box using the Polygon method in Google Maps

In [40]:
fig = gmaps.figure(center=sather_gate_berkeley, zoom_level=10)

five_mile_box = gmaps.Polygon(corners, stroke_color='blue', fill_color='blue')

drawing_layer = gmaps.drawing_layer(features=[five_mile_box])

fig.add_layer(drawing_layer)

fig

Figure(layout=FigureLayout(height='420px'))